# Surface Mapper Playground

This notebook tests creating a `Surface` object with `ow_to_sid()` using normalized SurfaceGrid test JSON input, then exports a flattened JSON representation including inherited attributes.

### Environment setup and imports

Import standard modules and project models/mappers. Paths are configured relative to the repository root so this notebook can run from the project checkout.

In [1]:
import json
from pathlib import Path

from pydantic import ValidationError

from dsis_model_sdk.models.common import SurfaceGrid

from mappers.surface_helpers import flatten_dict
from mappers.mappers_dsis import ow_to_sid, normalize_surfacegrid_payload
from models.interpretation import PipelineMetadata
from models.origin import SourceSystem, Project

In [2]:
repo_root = Path.cwd()
input_path = repo_root / "../tests/data/surfacegrid_volve_public.json"

### Load normalized SurfaceGrid test JSON

Load test JSON, apply normalization used in tests, and confirm critical keys exist.

In [3]:
raw_payload = json.loads(input_path.read_text(encoding="utf-8"))
normalized_payload = normalize_surfacegrid_payload(raw_payload)

required_keys = sorted(
    name for name, field in SurfaceGrid.model_fields.items() if field.is_required()
)

missing_required = [key for key in required_keys if key not in normalized_payload]
assert not missing_required, f"Missing required keys: {missing_required}"

surfacegrid = SurfaceGrid.model_validate(normalized_payload)
print("Loaded and validated SurfaceGrid")
print(
    f"native_uid={surfacegrid.native_uid}, name={surfacegrid.map_data_set_name}, crs={surfacegrid.crs}"
)

Loaded and validated SurfaceGrid
native_uid=2636, name=ihdTHugin13flt3, crs=ST_ED50_UTM31N_P23031_T1133


### Instantiate Surface from `ow_to_sid()`

Create pipeline metadata and run conversion. If mapper validation fails, the error is shown clearly.

In [4]:
pipeline_metadata = PipelineMetadata(id="11111111-1111-1111-1111-111111111111")
project = Project(
    source_system=SourceSystem.OPENWORKS,
    database="BG4FROST",
    name="VOLVE_PUBLIC",
    timezone="Europe/Berlin",
)

In [5]:
surface = None
try:
    surface = ow_to_sid(project, surfacegrid, pipeline_metadata)
    print(f"Created object: {type(surface).__name__}")
    print(surface.model_dump_json(indent=2))
except ValidationError as exc:
    print("ow_to_sid validation failed:")
    print(exc)
    raise

Created object: Surface
{
  "source": {
    "project": {
      "source_system": "OpenWorks R5000",
      "database": "BG4FROST",
      "name": "VOLVE_PUBLIC",
      "timezone": "Europe/Berlin",
      "last_pipeline_run_date": null
    },
    "native_uid": "2636",
    "name": "ihdTHugin13flt3",
    "crs": "ST_ED50_UTM31N_P23031_T1133",
    "z_domain": "TVDSS",
    "z_unit": "meters",
    "create_user": "IHD",
    "update_user": null,
    "remark": null,
    "create_date": "2013-11-15T08:20:49",
    "create_date_utc": null,
    "update_date": null,
    "update_date_utc": null,
    "ow": {
      "geo_name": "UNKNOWN",
      "geo_type": "SURFACE",
      "attribute": "DEPTH"
    },
    "petrel": null
  },
  "pipeline": {
    "id": "11111111-1111-1111-1111-111111111111",
    "create_date": null,
    "update_date": null,
    "file_availability": null,
    "deleted": null,
    "deleted_date": null
  },
  "extent": null,
  "collection": [],
  "geometry": {
    "ncol": 879,
    "nrow": 629,
    

### Flatten Surface object to JSON

Flatten nested dictionaries to dot-path keys for deterministic, compact comparison and downstream inspection.

In [6]:
surface_dict = surface.model_dump(mode="json", exclude_none=False)
flat_surface = dict(sorted(flatten_dict(surface_dict).items(), key=lambda kv: kv[0]))
print(f"Flattened keys: {len(flat_surface)}")
print(json.dumps(dict(list(flat_surface.items())), indent=2))

Flattened keys: 41
{
  "collection": [],
  "extent": null,
  "geometry.left_handed": true,
  "geometry.ncol": 879,
  "geometry.nrow": 629,
  "geometry.rotation": 0.0,
  "geometry.xinc": 12.0,
  "geometry.xori": 429588.0,
  "geometry.yinc": 12.0,
  "geometry.yori": 6475211.0,
  "nnan": null,
  "ntotal": null,
  "parent_surface_id": null,
  "pipeline.create_date": null,
  "pipeline.deleted": null,
  "pipeline.deleted_date": null,
  "pipeline.file_availability": null,
  "pipeline.id": "11111111-1111-1111-1111-111111111111",
  "pipeline.update_date": null,
  "source.create_date": "2013-11-15T08:20:49",
  "source.create_date_utc": null,
  "source.create_user": "IHD",
  "source.crs": "ST_ED50_UTM31N_P23031_T1133",
  "source.name": "ihdTHugin13flt3",
  "source.native_uid": "2636",
  "source.ow.attribute": "DEPTH",
  "source.ow.geo_name": "UNKNOWN",
  "source.ow.geo_type": "SURFACE",
  "source.petrel": null,
  "source.project.database": "BG4FROST",
  "source.project.last_pipeline_run_date": nu

### Notes

- This notebook uses the same normalization logic from `surface_helpers.py` as tests (normalizes `rotation_i`, `rotation_j`, `data_min`, `data_max`).
